In [ ]:
# FrozenEstimator, которым калибратор берет уже обученную модель, есть в scikit-learn начиная с 1.6
%pip install -q "scikit-learn>=1.6"

# **Калибровка вероятностей наивного Байеса**

### Туториал к уроку «Наивные Байесовские модели»

В уроке «Наивные Байесовские модели» мы сказали две вещи: вероятности наивного Байеса прижимаются к $0$ и $1$, поэтому самому числу верить нельзя, а порядок классов и направление вкладов признаков остаются осмысленными. Здесь проверим обе на данных и посмотрим, что делает с ними калибровка.

Калибровка — это монотонное преобразование скора модели, обученное на отдельной выборке так, чтобы среди объектов с прогнозом $p$ доля класса 1 была близка к $p$. Разберем два способа:

1. **калибровка Платта** — логистическая функция от исходной вероятности, $\hat p = \sigma(a\,p + b)$;
2. **изотоническая калибровка** — неубывающая ступенчатая функция, подобранная по данным.

И проверим, что калибровка меняет, а что нет: число вероятности, порядок объектов, решение по порогу и вклады признаков.

Заданий в тренажере по этому туториалу нет.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42

## Данные

Берем тот же набор Pima Indians Diabetes, что и в уроке «Случайный лес: Практика»: восемь числовых признаков, целевая переменная `Outcome` — есть ли у пациентки диабет. Калибратор нельзя учить на тех же строках, что и модель: на обучающих данных ее вероятности смещены сильнее, чем на новых. Поэтому выборку делим на три части.

In [ ]:
path = 'https://github.com/SadSabrina/open-xai-materials/raw/refs/heads/main/data/diabetes.csv'
data = pd.read_csv(path)
X = data.drop(columns='Outcome')
y = data['Outcome']

# половина — на обучение модели, четверть — на обучение калибратора, четверть — на проверку
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.5, random_state=RANDOM_STATE, stratify=y)
X_cal, X_test, y_cal, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=RANDOM_STATE, stratify=y_rest)
print('обучение:', len(X_train), ' калибровка:', len(X_cal), ' проверка:', len(X_test))

## Наивный Байес и его вклады

Гауссовский наивный Байес хранит для каждого класса $c$ и признака $i$ среднее $\mu_{c,i}$ и дисперсию $\sigma^2_{c,i}$. Как в уроке, прогноз раскладывается в сумму на логарифмической шкале; для двух классов удобно записать ее как log-odds:

$$\log\frac{P(1\mid x)}{P(0\mid x)} = \log\frac{P(1)}{P(0)} + \sum_{i=1}^{n}\Big[\log P(x_i\mid 1) - \log P(x_i\mid 0)\Big].$$

Слагаемое в квадратных скобках — вклад признака $i$. Проверим, что сигмоида от суммы вкладов в точности дает `predict_proba`.

In [ ]:
nb = GaussianNB().fit(X_train, y_train)

def contributions(model, X):
    """Вклады признаков в log-odds класса 1: log P(x_i | 1) - log P(x_i | 0) для гауссовского NB."""
    x = X.values[:, None, :]
    log_lik = -0.5 * np.log(2 * np.pi * model.var_)[None] - 0.5 * (x - model.theta_[None]) ** 2 / model.var_[None]
    return pd.DataFrame(log_lik[:, 1, :] - log_lik[:, 0, :], columns=X.columns, index=X.index)

prior = np.log(nb.class_prior_[1] / nb.class_prior_[0])
contrib = contributions(nb, X_test)
log_odds = prior + contrib.sum(axis=1)

p_raw = nb.predict_proba(X_test)[:, 1]
print('априорный член log P(1)/P(0):', round(prior, 4))
print('наибольшее расхождение sigmoid(сумма вкладов) и predict_proba:',
      np.abs(1 / (1 + np.exp(-log_odds)) - p_raw).max())

Расхождение на уровне ошибки округления: разложение точное, и именно его мы читаем, когда интерпретируем наивный Байес.

## Насколько вероятностям можно верить

Кривая надежности сравнивает предсказанную вероятность с тем, что случилось на самом деле. Разобьем проверочные объекты на пять равных по числу групп по предсказанной вероятности и в каждой группе сравним среднюю предсказанную вероятность с долей класса 1. У откалиброванной модели два столбца совпадают.

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, p_raw, n_bins=5, strategy='quantile')
print(pd.DataFrame({'средняя предсказанная вероятность': mean_pred.round(3),
                    'доля класса 1 на самом деле': frac_pos.round(3)}))
print()
print('прогнозов ниже 0.01 или выше 0.99:', int(((p_raw < 0.01) | (p_raw > 0.99)).sum()), 'из', len(p_raw))

В нижней группе прогноз и доля класса 1 почти совпадают, а выше расходятся: в верхней группе модель обещает в среднем больше $0.9$, а класс 1 там примерно у двух третей объектов, в средних группах она, наоборот, занижает. Вероятности разнесены к краям сильнее, чем это оправдывают данные, — это то самое прижимание к $0$ и $1$ из урока; часть прогнозов уходит за $0.01$ и $0.99$.

## Платт и изотоническая калибровка

`CalibratedClassifierCV` учит калибратор поверх модели. Обертка `FrozenEstimator` говорит ему не переобучать наивный Байес, а взять уже обученный: так калибратор преобразует скор **одной и той же** модели, и сравнение до и после остается сравнением одной модели. Калибратор учится на калибровочной части.

In [ ]:
platt = CalibratedClassifierCV(FrozenEstimator(nb), method='sigmoid').fit(X_cal, y_cal)
isotonic = CalibratedClassifierCV(FrozenEstimator(nb), method='isotonic').fit(X_cal, y_cal)

p_platt = platt.predict_proba(X_test)[:, 1]
p_iso = isotonic.predict_proba(X_test)[:, 1]

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], linestyle='--', color='grey', label='perfect calibration')
for name, p, color in [('naive Bayes, raw', p_raw, '#eb6834'),
                       ('Platt (sigmoid)', p_platt, '#2a78d6'),
                       ('isotonic', p_iso, '#1a9e5f')]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=5, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', color=color, label=name)
ax.set_xlabel('mean predicted probability of class 1')
ax.set_ylabel('actual share of class 1')
ax.set_title('Reliability curve on the test split, 5 quantile bins')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

Обе калиброванные кривые в целом ближе к диагонали, чем исходная, но не в каждой группе: калибровка Платта — сигмоида с двумя параметрами, и, опустив верхние группы, она поднимает нижнюю выше доли класса 1 в ней. Сравним числами: Brier и log-loss меряют качество самих вероятностей (меньше — лучше), ROC-AUC — только порядок объектов.

In [ ]:
rows = {}
for name, p in [('без калибровки', p_raw), ('Платт', p_platt), ('изотоническая', p_iso)]:
    rows[name] = {
        'Brier': round(brier_score_loss(y_test, p), 4),
        'log-loss': round(log_loss(y_test, p), 4),
        'ROC-AUC': round(roc_auc_score(y_test, p), 4),
        'разных значений': len(np.unique(p)),
        'Спирмен с исходным': round(spearmanr(p_raw, p).statistic, 4),
        'сменили класс при 0.5': int(((p >= 0.5) != (p_raw >= 0.5)).sum()),
    }
pd.DataFrame(rows).T

Что видно в таблице:

- **Платт** — строго возрастающая функция исходной вероятности. Порядок объектов не меняется (корреляция Спирмена с исходной вероятностью равна 1), поэтому и ROC-AUC тот же. Меняются сами числа: Brier и log-loss падают.
- **Изотоническая** — неубывающая ступенчатая функция. Разных значений вероятности становится столько, сколько ступенек, объекты внутри одной ступеньки получают одинаковую вероятность, и порядок между ними теряется — корреляция Спирмена меньше 1, ROC-AUC может немного сдвинуться. Крайние ступеньки выдают вероятности $0$ или $1$, и log-loss, который за уверенную ошибку штрафует сильнее всего, может оказаться хуже исходного. Это ступенчатые артефакты изотонической калибровки, особенно заметные на небольшой калибровочной выборке.
- Часть объектов **сменила класс** при пороге $0.5$. Значит ли это, что калибровка сдвинула границу решения? Проверим.

In [ ]:
sig = platt.calibrated_classifiers_[0].calibrators[0]
t = -sig.b_ / sig.a_            # исходная вероятность, которую Платт переводит ровно в 0.5
print('порог 0.5 после Платта = порог', round(t, 4), 'на исходной вероятности')
print('решения совпадают на всей проверке:', bool(((p_platt >= 0.5) == (p_raw >= t)).all()))

Решения совпадают: порог $0.5$ на калиброванной вероятности — это другой порог на исходной. Монотонное преобразование не создает новой границы решения, оно переименовывает пороги: любая граница калиброванной модели есть граница исходной при каком-то пороге. Сменившие класс объекты — это объекты между двумя порогами.

## Что калибровка делает с вкладами признаков

Калибратор смотрит только на итоговый скор, признаков он не видит. Поэтому вклады $\log P(x_i\mid 1) - \log P(x_i\mid 0)$ остаются теми же, что у исходной модели, — и их порядок, и знаки. Но сумма вкладов складывается в исходную вероятность, а не в калиброванную. Посмотрим на одном объекте проверки.

In [ ]:
i = X_test.index[0]
row = contrib.loc[i]
print(row.sort_values(key=abs, ascending=False).round(3).to_string())
print()
print('априорный член + сумма вкладов (log-odds):', round(log_odds.loc[i], 4))
print('sigmoid(log-odds)                         :', round(1 / (1 + np.exp(-log_odds.loc[i])), 4))
print('исходная вероятность predict_proba        :', round(p_raw[0], 4))
print('после Платта                              :', round(p_platt[0], 4))
print('после изотонической                       :', round(p_iso[0], 4))
print()
gap = np.abs(1 / (1 + np.exp(-log_odds.values)) - p_platt)
print('на всей проверке: sigmoid(сумма вкладов) отличается от вероятности Платта в среднем на',
      round(gap.mean(), 4), ', наибольшее расхождение', round(gap.max(), 4))

## Что стоит унести

1. Калибровка Платта и изотоническая — монотонные преобразования скора поверх уже обученной модели. Они исправляют число вероятности, и Brier на проверке падает.
2. Порядок объектов калибровка сохраняет (у Платта строго, у изотонической с точностью до ступенек), а граница решения после калибровки — граница исходной модели при другом пороге.
3. Аддитивное разложение на шкале вероятности ломается: вклады признаков объясняют исходную вероятность, и сумма вкладов больше не равна откалиброванному прогнозу.
4. Неверную атрибуцию калибровка не чинит: вклады признаков остаются вкладами исходной модели, со всеми последствиями нарушенного предположения независимости. Если вкладам нельзя было верить до калибровки, им нельзя верить и после.
5. Изотоническая калибровка дает ступенчатые артефакты: одинаковые вероятности у разных объектов и вероятности $0$ и $1$ на крайних ступеньках.